# Xiangqi — Tournament trên Colab CPU

Chọn `BUDGET_MODE` để chạy một trong hai thực nghiệm:

- `equal_time`: các agent tìm kiếm có cùng ngân sách **mỗi nước** (mặc định 1 giây).
  Đây là giới hạn mềm tùy agent; `random`, `rules`, `tabular_q` không tìm kiếm.
- `default`: từng agent giữ cấu hình mặc định được phát hành, theo `tournament_default.yaml`.
Notebook **không đặt thời hạn cho cả giải** và không thay đổi thuật toán hay lịch đấu.

- Code và model chạy trên ổ tạm `/content`; log từng ván ghi thẳng vào Google Drive.
- Sau khi hoàn thành hoặc dừng cell có xử lý, tự sinh CSV, biểu đồ, `report.md` và ZIP lên Drive.
- Chạy lại cùng `SESSION_NAME` để resume. Ván chưa hoàn tất phải đánh lại từ đầu.
- Drive có thể đồng bộ trễ/lỗi; runtime bị thu hồi đột ngột không thể chạy bước xuất cuối.
  Log đã lưu vẫn dùng được để resume hoặc tạo báo cáo sau khi kết nối lại.

Chọn **Runtime → Change runtime type → Hardware accelerator: None**.
Chỉ mở một runtime ghi vào một session. Không dùng SSH/keep-alive để vượt giới hạn Colab.
[FAQ Colab](https://research.google.com/colaboratory/faq.html) hạn chế SSH và “chess training”
trên runtime miễn phí không có compute units; FAQ không làm rõ việc đánh giá Xiangqi có thuộc
mục này không. Kiểm tra chính sách/gói dịch vụ phù hợp trước khi chạy.

## 1. Cấu hình

Chọn `BUDGET_MODE = 'equal_time'` hoặc `'default'`; notebook tự chọn YAML tương ứng.
Với `MODE = 'full'`, thư mục Drive lần lượt là `equal_time_01` và `default_01`.
`SESSION_NAME` là tên cố định, không tự tăng số: chạy lại cùng tên sẽ resume.
Muốn thực nghiệm mới cùng budget mode, đổi hậu tố `01` thành `02` hoặc đặt tên riêng.
Giữ `MODE = 'full'` cho giải đầy đủ; đây là lựa chọn riêng với `BUDGET_MODE`.

Mặc định để `TABULAR_Q_DRIVE_PATH` trống: dùng `models/tabular_q/q_table.json.gz` đã có trong repo.
Chỉ điền đường dẫn tới **file** `q_table.json.gz` trên Drive nếu muốn dùng checkpoint khác.
Nếu chủ động không đánh giá tabular Q, đặt `INCLUDE_TABULAR_Q = False`; số agent/ván sẽ giảm.
Neural MCTS mặc định được tải bằng script kiểm SHA-256 có sẵn trong repo.
Thiếu checkpoint của agent được chọn sẽ báo lỗi, không tự bỏ agent.

`WORKERS = CPU_COUNT` tự dùng số CPU runtime báo, có xét giới hạn affinity nếu có.
Runtime 2 vCPU sẽ chọn 2 worker; có thể đặt `WORKERS = 1` hoặc số khác để điều chỉnh.
Số CPU báo về không đảm bảo tài nguyên tính toán độc quyền; tăng worker không đảm bảo tăng tốc tuyến tính.
`WORKERS = 5` là 5 ván song song, không phải 5 giây mỗi nước. Tránh 5 worker trên 2 vCPU.
Đổi worker được phép khi resume: Stop cell chạy, đợi subprocess dừng, sửa worker rồi chạy lại
cell cấu hình và cell chạy với cùng `SESSION_NAME`. Không đổi budget/config/model.
Các ván đã lưu được giữ lại; ván dở đánh lại. Thay đổi worker được ghi trong `manifest.json`.
Agent giới hạn bằng thời gian có thể chọn nước khác khi mức cạnh tranh CPU thay đổi.
Dùng `MODE = 'smoke'` và tên session riêng để kiểm tra setup trước; `full` chạy giải thật,
`pilot` ước lượng thời gian. Đổi cấu hình/mode/danh sách agent cần **tên session mới**.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import signal
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

REPO_URL = 'https://github.com/Ko4rut/ML-Advance-Group.git'
SOURCE_REF = 'feature/tournament'  # branch/tag/commit cho session mới
REPO_DIR = Path('/content/ML-Advance-Group-tournament')
BUDGET_MODE = 'equal_time'  # equal_time | default
MODE = 'full'  # full | smoke | pilot
CONFIG_RELATIVE = f'configs/tournament_{BUDGET_MODE}.yaml'
SESSION_NAME = f'{BUDGET_MODE}_01' if MODE == 'full' else f'{BUDGET_MODE}_{MODE}_01'
DRIVE_ROOT = Path('/content/drive/MyDrive/XiangqiAI/tournaments')
# Dùng số CPU runtime báo và giới hạn affinity nếu có; có thể đặt số cụ thể.
CPU_COUNT = os.cpu_count() or 1
if hasattr(os, 'sched_getaffinity'):
    try:
        CPU_COUNT = min(CPU_COUNT, len(os.sched_getaffinity(0)))
    except OSError:
        pass
CPU_COUNT = max(1, CPU_COUNT)
WORKERS = CPU_COUNT  # runtime 2 vCPU -> 2 worker; đổi số để điều chỉnh thủ công
INCLUDE_NEURAL_MCTS = True
INCLUDE_TABULAR_Q = True
TABULAR_Q_DRIVE_PATH = ''  # trống = checkpoint trong repo; điền Drive path để chọn bản khác
DOWNLOAD_ZIP = True  # ZIP luôn được lưu trên Drive; tùy chọn tải thêm về máy

if BUDGET_MODE not in {'equal_time', 'default'}:
    raise ValueError('BUDGET_MODE phải là equal_time hoặc default.')
if not SESSION_NAME or Path(SESSION_NAME).name != SESSION_NAME or SESSION_NAME in {'.', '..'}:
    raise ValueError('SESSION_NAME phải là tên một thư mục, không chứa đường dẫn.')
if MODE not in {'full', 'smoke', 'pilot'}:
    raise ValueError('MODE phải là full, smoke hoặc pilot.')
if isinstance(WORKERS, bool) or not isinstance(WORKERS, int) or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
SESSION_DIR = DRIVE_ROOT / SESSION_NAME
BACKEND_DIR = REPO_DIR / 'project' / 'backend'
print('Python:', sys.version.split()[0], '| CPU logic:', os.cpu_count(), '| CPU khả dụng:', CPU_COUNT)
print('Session:', SESSION_DIR, '| Budget:', BUDGET_MODE, '| Mode:', MODE, '| Workers:', WORKERS)
if WORKERS > CPU_COUNT:
    print('LƯU Ý: workers vượt CPU khả dụng, các ván tranh CPU và agent có thể tìm kiếm ít hơn mỗi nước.')

## 2. Kết nối Drive

Chỉ chạy notebook/source bạn tin cậy: mã chạy trong runtime có thể truy cập Drive đã mount.
Thư mục session chứa `source.json`, bản cấu hình, phiên bản thư viện và `runs/<run_id>/`.
ZIP nằm cạnh thư mục session, tránh nén lặp ZIP vào chính nó.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
SESSION_DIR.mkdir(parents=True, exist_ok=True)
print('Drive ready:', SESSION_DIR)

## 3. Tải source và cài thư viện

Session ghi nhớ commit ở lần setup đầu. Khi runtime mới được cấp, notebook tải lại **đúng commit đó**,
không tự nâng code khi resume. `SOURCE_REF` chỉ chọn code cho session mới.
Notebook không reset/pull một checkout đang có thay đổi. Repo cần truy cập được từ Colab;
nếu dùng repo riêng tư, chuẩn bị quyền truy cập hoặc checkout sẵn, không ghi token vào notebook.

In [ ]:
source_file = SESSION_DIR / 'source.json'
saved_source = json.loads(source_file.read_text()) if source_file.exists() else None
if saved_source and saved_source['repo_url'] != REPO_URL:
    raise ValueError('Repo URL khác session cũ; dùng SESSION_NAME mới.')
target_ref = saved_source['commit'] if saved_source else SOURCE_REF
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
origin_url = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin_url != REPO_URL:
    raise ValueError('Checkout đang dùng repo khác; chọn REPO_DIR mới.')
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout đang có thay đổi; dùng REPO_DIR sạch để bảo toàn source.')
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', target_ref], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/evaluation/tournament.py').is_file():
    raise FileNotFoundError('Source đã chọn chưa có tournament; kiểm tra SOURCE_REF.')
if saved_source is None:
    source_file.write_text(json.dumps({'repo_url': REPO_URL, 'commit': commit}, indent=2) + '\n')
print('Source commit:', commit)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(BACKEND_DIR),
    'PyYAML>=6,<7', 'python-dotenv>=1,<2', 'matplotlib>=3.6,<4',
], check=True)
if INCLUDE_NEURAL_MCTS:
    # Dùng torch có sẵn nếu import được; chỉ cài CPU wheel khi chưa có.
    probe = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'])
    if probe.returncode:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--index-url',
            'https://download.pytorch.org/whl/cpu', 'torch>=2.11,<2.12',
        ], check=True)
os.environ['XIANGQI_NEURAL_DEVICE'] = 'cpu'
os.environ['XIANGQI_NEURAL_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/xiangqi-matplotlib'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))


## 4. Chuẩn bị checkpoint và chốt cấu hình

Tabular Q mặc định dùng checkpoint đã được Git theo dõi trong repo. Nếu chọn file trên Drive,
notebook copy file đó vào ổ tạm để worker không phải tải nhiều lần qua Drive.
Bản cấu hình cố định lưu trên Drive; notebook kiểm hash/config khi resume.
Đường dẫn model có thể thay đổi giữa máy, nhưng nội dung checkpoint phải giữ nguyên.
Khi chọn đủ 8 agent, `full` có 28 cặp × 100 ván = 2.800 ván theo config hiện tại.

In [ ]:
def prepare_tabular_q(backend_dir, drive_path):
    from src.training.tabular_qlearning import TabularQTable

    if drive_path:
        source_q = Path(drive_path).expanduser()
        if not source_q.is_file():
            raise FileNotFoundError(f'Không có checkpoint tại Drive path đã chọn: {source_q}')
        # Validate before copying; an explicit override never falls back to the repo.
        TabularQTable.load(source_q)
        local_q = Path('/content/xiangqi-tournament-models/q_table.json.gz')
        local_q.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source_q, local_q)
    else:
        local_q = backend_dir / 'models/tabular_q/q_table.json.gz'
        if not local_q.is_file():
            raise FileNotFoundError(
                'Source đã chọn thiếu models/tabular_q/q_table.json.gz; '
                'chọn đúng SOURCE_REF hoặc điền TABULAR_Q_DRIVE_PATH.'
            )
    TabularQTable.load(local_q)
    print('Tabular Q checkpoint:', local_q)
    return {'checkpoint': str(local_q), 'required': True}


In [ ]:
import yaml

raw = yaml.safe_load((BACKEND_DIR / CONFIG_RELATIVE).read_text(encoding='utf-8'))
if raw.get('budget_mode', 'default') != BUDGET_MODE:
    raise ValueError('Budget mode trong YAML không khớp BUDGET_MODE đã chọn.')
if INCLUDE_NEURAL_MCTS:
    subprocess.run([
        sys.executable, 'scripts/fetch_neural_checkpoint.py', '--model', 'distill-v4-large',
    ], cwd=BACKEND_DIR, check=True)
    raw['agents']['neural_mcts'].update(required=True, model='distill-v4-large')
else:
    raw['agents'].pop('neural_mcts', None)
if INCLUDE_TABULAR_Q:
    raw['agents']['tabular_q'] = prepare_tabular_q(BACKEND_DIR, TABULAR_Q_DRIVE_PATH)
else:
    raw['agents'].pop('tabular_q', None)
# Ghi file ngoài checkout; không sửa cấu hình gốc của repo.
candidate_config = Path('/content/xiangqi-tournament-candidate.yaml')
candidate_config.write_text(yaml.safe_dump(raw, sort_keys=False), encoding='utf-8')
print('Agents:', ', '.join(raw['agents']))
if BUDGET_MODE == 'equal_time':
    print('Thời gian mỗi nước tìm kiếm:', raw['time_per_move'], 'giây.')
else:
    print('Budget default: từng agent giữ cấu hình mặc định; không áp cùng thời gian mỗi nước.')
print('Không có thời hạn toàn giải; log và report lưu trên Drive:', SESSION_DIR)
versions = subprocess.check_output([sys.executable, '-m', 'pip', 'list', '--format=json'], text=True)
(SESSION_DIR / 'packages.json').write_text(versions, encoding='utf-8')


## 5. Chạy, resume và tự xuất kết quả

Cell định nghĩa hàm bên dưới dùng CLI của repo, không tự triển khai lại tournament.
Cell chạy tiếp theo sẽ:

1. Kiểm tra config/commit/mode và checkpoint; khóa config cho session.
2. Tạo giải mới hoặc tự resume thư mục run duy nhất của session.
3. In log trực tiếp; `games.jsonl` ghi + flush sau **mỗi ván hoàn tất** lên Drive.
4. Sau khi chạy hoặc dừng có xử lý, tạo `summary/` và ZIP toàn session, kèm SHA-256.

Nút Stop gửi yêu cầu dừng và chờ subprocess kết thúc trước khi xuất file.
Dừng cưỡng bức/thu hồi VM không thể bảo đảm chạy cleanup; khi kết nối lại, chạy setup rồi resume,
hoặc chỉ chạy cell xuất lại ở cuối. Không đổi cấu hình/commit hay chạy hai runtime chung session.
Không lấy báo cáo giải chưa hoàn tất làm kết quả của một giải cân bằng đã hoàn tất.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def checked_cli(backend_dir, *args):
    return subprocess.run(
        [sys.executable, '-u', '-m', 'src', *map(str, args)], cwd=backend_dir, check=True,
    )


def session_run_dir(session_dir):
    runs = sorted((session_dir / 'runs').glob('*/manifest.json'))
    if len(runs) > 1:
        raise RuntimeError('Session có nhiều run; dùng session riêng cho từng giải.')
    return runs[0].parent if runs else None


def export_session(backend_dir, session_dir):
    run_dir = session_run_dir(session_dir)
    if run_dir is None:
        raise FileNotFoundError('Chưa có manifest để xuất; kiểm tra lỗi setup/tournament ở trên.')
    # Nếu report lỗi, vẫn giữ ZIP raw logs; không che giấu lỗi report.
    report_error = None
    try:
        checked_cli(backend_dir, 'tournament-report', run_dir)
    except subprocess.CalledProcessError as exc:
        report_error = str(exc)
        print('REPORT FAILED; ZIP vẫn chứa raw logs:', exc)
    status = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))['status']
    print('Trạng thái giải:', status, '| Run:', run_dir.name)
    (session_dir / 'export.json').write_text(
        json.dumps({'run_id': run_dir.name, 'status': status, 'report_error': report_error}, indent=2) + '\n',
        encoding='utf-8',
    )
    # Nén trên ổ tạm, copy file hoàn chỉnh rồi rename trên Drive.
    archive_path = session_dir.parent / (session_dir.name + '.zip')
    with tempfile.TemporaryDirectory(prefix='xiangqi-export-') as temporary:
        local_zip = Path(temporary) / archive_path.name
        with zipfile.ZipFile(local_zip, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
            for path in sorted(session_dir.rglob('*')):
                if path.is_file():
                    archive.write(path, path.relative_to(session_dir.parent))
        with zipfile.ZipFile(local_zip) as archive:
            if archive.testzip() is not None:
                raise IOError('ZIP bị lỗi; raw logs vẫn ở Drive.')
        digest = sha256_file(local_zip)
        staging = archive_path.with_suffix('.zip.uploading')
        shutil.copyfile(local_zip, staging)
        copied_digest = sha256_file(staging)
        if copied_digest != digest:
            raise IOError('Checksum ZIP trên Drive không khớp; giữ archive cũ nếu có.')
        os.replace(staging, archive_path)
    archive_path.with_suffix('.zip.sha256').write_text(digest + '  ' + archive_path.name + '\n')
    print('ZIP:', archive_path, '| SHA-256:', digest)
    return {'run_dir': run_dir, 'archive': archive_path, 'status': status, 'report_error': report_error}


def run_and_export(backend_dir, session_dir, candidate, workers, mode):
    from src.evaluation import tournament as tournament

    session_dir.mkdir(parents=True, exist_ok=True)
    frozen_config = session_dir / 'tournament.input.yaml'
    config = tournament.resolve_config(tournament.load_raw_config(candidate), mode=mode)
    metadata_path = session_dir / 'experiment.json'
    settings = {'mode': mode, 'config_hash': config.config_hash}
    if metadata_path.exists():
        if json.loads(metadata_path.read_text()) != settings:
            raise ValueError('Config/model/mode khác session cũ; giữ nguyên hoặc dùng SESSION_NAME mới.')
    else:
        if session_run_dir(session_dir) is not None:
            raise RuntimeError('Session đã có run nhưng thiếu experiment.json; kiểm tra trước khi resume.')
        shutil.copyfile(candidate, frozen_config)
        metadata_path.write_text(json.dumps(settings, indent=2) + '\n')
    # Re-validate the saved input, including optional checkpoint hashes.
    saved = tournament.resolve_config(tournament.load_raw_config(frozen_config), mode=mode)
    if saved.config_hash != config.config_hash:
        raise ValueError('Cấu hình đã lưu bị thay đổi; không tiếp tục session này.')
    run_dir = session_run_dir(session_dir)
    command = [sys.executable, '-u', '-m', 'src', 'tournament', '--config', str(frozen_config),
               '--workers', str(workers)]
    if run_dir is None:
        command += ['--output-root', str(session_dir / 'runs')]
        if mode != 'full':
            command += ['--' + mode]
    else:
        command += ['--resume', str(run_dir)]
    print('Command:', ' '.join(command), flush=True)
    print('Không có thời hạn toàn giải; ngân sách mỗi nước lấy từ config.', flush=True)
    process = subprocess.Popen(command, cwd=backend_dir, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        print('Đang dừng tournament và giữ các ván đã hoàn tất...', flush=True)
        process.send_signal(signal.SIGINT)
        try:
            tail, _ = process.communicate(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            tail, _ = process.communicate()
            print('Phải dừng cưỡng bức; có thể cần resume để sửa dòng log cuối.')
        print(tail, end='', flush=True)
        returncode = process.returncode
    finally:
        # Do not archive while a child could still be appending to games.jsonl.
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        process.stdout.close()
    result = export_session(backend_dir, session_dir)
    if returncode not in (0, 130):
        raise RuntimeError(f'Tournament exit={returncode}; raw logs/ZIP ở {session_dir}.')
    if result['report_error']:
        raise RuntimeError('Raw logs đã được nén nhưng report lỗi; xem output và xuất lại.')
    return result

In [ ]:
# Chạy lại cell này sẽ resume, không tạo giải mới nếu session đã có manifest.
# Import từ checkout đã chọn (không import module trước khi bước setup hoàn tất).
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
result = run_and_export(BACKEND_DIR, SESSION_DIR, candidate_config, WORKERS, MODE)
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(result['archive']))

## 6. Xuất lại / xem kết quả (không chạy thêm ván)

Cell này tùy chọn; đặt `EXPORT_AGAIN = True` khi cần xuất lại.
Mặc định cell được bỏ qua để **Run all** không xuất/tải ZIP hai lần.
Sau reconnect, chạy các cell setup và định nghĩa hàm rồi chạy cell dưới nếu chỉ muốn xuất lại.
Nếu có dòng JSONL cuối bị cắt, report có thể cảnh báo/bỏ qua nó; resume sẽ sửa log trước khi đánh tiếp.
`status=running` có thể còn sót sau khi VM bị thu hồi; không có nghĩa giải đã hoàn tất.
Khi lỗi Drive/quota, khắc phục kết nối/dung lượng rồi xuất lại, không xóa raw logs.
ZIP chứa toàn bộ session: log từng nước của ván đã hoàn tất, lỗi agent, metadata,
cấu hình, CSV, biểu đồ PNG và báo cáo. File plot tùy chỉnh trên máy bạn không nằm trong source
và không được tự mang sang Colab.

In [ ]:
EXPORT_AGAIN = False  # đổi thành True khi chỉ muốn xuất lại kết quả
if EXPORT_AGAIN:
    result = export_session(BACKEND_DIR, SESSION_DIR)
    from IPython.display import Markdown, display
    report_path = result['run_dir'] / 'summary/report.md'
    if report_path.exists() and not result['report_error']:
        display(Markdown(report_path.read_text(encoding='utf-8')))
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(str(result['archive']))